In [0]:
%sql
-- ============================================================================
-- BEST PRACTICE #4: Centralize Shared Dimensions in Canonical Dimension MVs
-- ============================================================================
--
-- "Canonical" = the ONE official, authoritative definition that everyone 
-- defers to. Instead of redefining conformed dimensions inside each fact 
-- metric view's YAML, create a canonical dimension METRIC VIEW that all 
-- fact MVs join to.
--
-- ============================================================================
-- SCHEMA USED:   home_dipankar_kushari.metric_view
-- SOURCE DATA:   samples.tpch
-- DBR VERSION:   17.3 (uses `dimensions` keyword; `fields` available in 18.1+)
-- ============================================================================
--
-- TPC-H Schema for this example:
--
--   region ←── nation ←── supplier ──→ lineitem (fact)
--                                  ──→ partsupp (fact)
--
-- Conformed dimensions: region, nation, supplier
-- Facts sharing those dimensions: lineitem, partsupp
--
-- ============================================================================
-- WHAT WE'LL BUILD:
--
--   Layer 1 (Canonical Dim MVs):
--     • region_dim_mv      → source: samples.tpch.region
--     • nation_dim_mv      → source: samples.tpch.nation, joins to region_dim_mv
--     • supplier_dim_mv    → source: samples.tpch.supplier, joins to nation_dim_mv
--
--   Layer 2 (Fact MVs joining to canonical dim MVs):
--     • lineitem_fact_mv   → source: samples.tpch.lineitem, joins to supplier_dim_mv
--     • partsupp_fact_mv   → source: samples.tpch.partsupp, joins to supplier_dim_mv
--
--   Layer 3 (Combined multi-fact MV):
--     • supplier_orders_vs_supply_mv → source: supplier_dim_mv (MV as source!)
--                                      joins: lineitem + partsupp (one_to_many)
-- ============================================================================

SELECT 'Best Practice #4: Canonical Dimension MVs — Real Example' AS title,
       'home_dipankar_kushari.metric_view' AS target_schema,
       'samples.tpch' AS source_data

title,target_schema,source_data
Best Practice #4: Canonical Dimension MVs — Real Example,home_dipankar_kushari.metric_view,samples.tpch


In [0]:
%sql
-- ============================================================================
-- LAYER 1: CANONICAL DIMENSION METRIC VIEWS (defined ONCE)
-- ============================================================================
-- These are dimension-only MVs (no measures). They are the single source of
-- truth for all dimension metadata: comments, display_names, synonyms.
-- Every fact MV that needs this dimension joins HERE — never redefines it.
-- ============================================================================

-- ┌─────────────────────────────────────────────────────────────────────────┐
-- │  CANONICAL DIMENSION: region_dim_mv                                     │
-- │  The top of the geographic snowflake. 5 rows.                           │
-- └─────────────────────────────────────────────────────────────────────────┘

CREATE OR REPLACE VIEW home_dipankar_kushari.metric_view.region_dim_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Canonical conformed dimension: Geographic regions.
  Defined ONCE. All metric views needing region context join to this MV.
source: samples.tpch.region
dimensions:
  - name: region_key
    expr: r_regionkey
    comment: "Primary key for region"
  - name: region_name
    expr: r_name
    comment: "Geographic region (AFRICA, AMERICA, ASIA, EUROPE, MIDDLE EAST)"
    display_name: "Region"
    synonyms:
      - "geographic region"
      - "continent"
      - "world region"
$$

result


In [0]:
%sql
-- ┌─────────────────────────────────────────────────────────────────────────┐
-- │  CANONICAL DIMENSION: nation_dim_mv                                     │
-- │  Snowflakes to region_dim_mv (joins to another MV!). 25 rows.           │
-- └─────────────────────────────────────────────────────────────────────────┘

CREATE OR REPLACE VIEW home_dipankar_kushari.metric_view.nation_dim_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Canonical conformed dimension: Nations/countries.
  Snowflakes to region_dim_mv for geographic hierarchy.
  Defined ONCE — all MVs needing nation context join here.
source: samples.tpch.nation
joins:
  - name: region
    source: home_dipankar_kushari.metric_view.region_dim_mv
    on: "region.region_key = source.n_regionkey"
dimensions:
  - name: nation_key
    expr: n_nationkey
    comment: "Primary key for nation"
  - name: nation_name
    expr: n_name
    comment: "Country name (25 nations across 5 regions)"
    display_name: "Nation"
    synonyms:
      - "country"
      - "country name"
  - name: region_name
    expr: region.region_name
    comment: "Geographic region — inherited from region_dim_mv via snowflake join"
    display_name: "Region"
$$

result


In [0]:
%sql
-- ┌─────────────────────────────────────────────────────────────────────────┐
-- │  CANONICAL DIMENSION: supplier_dim_mv                                   │
-- │  Snowflakes to nation_dim_mv → region_dim_mv. 50,000 rows.             │
-- │  SHARED by both lineitem and partsupp facts.                            │
-- └─────────────────────────────────────────────────────────────────────────┘

CREATE OR REPLACE VIEW home_dipankar_kushari.metric_view.supplier_dim_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Canonical conformed dimension: Suppliers.
  Snowflakes to nation_dim_mv for geographic context.
  SHARED by lineitem and partsupp facts — defined ONCE here.
source: samples.tpch.supplier
joins:
  - name: nation
    source: home_dipankar_kushari.metric_view.nation_dim_mv
    on: "nation.nation_key = source.s_nationkey"
dimensions:
  - name: supplier_key
    expr: s_suppkey
    comment: "Primary key for supplier"
  - name: supplier_name
    expr: s_name
    comment: "Supplier name"
    display_name: "Supplier"
    synonyms:
      - "vendor"
      - "supplier name"
  - name: supplier_nation
    expr: nation.nation_name
    comment: "Supplier's country — from nation_dim_mv"
    display_name: "Supplier Nation"
  - name: supplier_region
    expr: nation.region_name
    comment: "Supplier's region — from nation_dim_mv → region_dim_mv snowflake"
    display_name: "Supplier Region"
  - name: account_balance
    expr: s_acctbal
    comment: "Supplier account balance"
    display_name: "Account Balance"
$$

result


In [0]:
%sql
-- ============================================================================
-- Verify Layer 1: Query the canonical dimension MVs
-- ============================================================================

-- Query supplier_dim_mv — it resolves the full snowflake chain automatically!
SELECT supplier_name, supplier_nation, supplier_region, account_balance
FROM home_dipankar_kushari.metric_view.supplier_dim_mv
LIMIT 10

supplier_name,supplier_nation,supplier_region,account_balance
Supplier#000029618,IRAN,MIDDLE EAST,5921.84
Supplier#000029619,SAUDI ARABIA,MIDDLE EAST,9339.38
Supplier#000029620,EGYPT,MIDDLE EAST,5968.25
Supplier#000029621,SAUDI ARABIA,MIDDLE EAST,8418.47
Supplier#000029622,IRAQ,MIDDLE EAST,5339.57
Supplier#000029623,FRANCE,EUROPE,2195.37
Supplier#000029624,VIETNAM,ASIA,1578.50
Supplier#000029625,MOZAMBIQUE,AFRICA,6882.52
Supplier#000029626,RUSSIA,EUROPE,7532.11
Supplier#000029627,UNITED STATES,AMERICA,2288.41


In [0]:
%sql
-- ============================================================================
-- LAYER 2: FACT METRIC VIEWS (join to the canonical dimension MVs)
-- ============================================================================
-- Key point: joins.source points to the METRIC VIEW, NOT the raw table!
-- The fact MV inherits ALL dimension metadata from the canonical MV.
-- NO dimension metadata is redefined here.
-- ============================================================================

-- ┌─────────────────────────────────────────────────────────────────────────┐
-- │  FACT MV 1: lineitem_fact_mv                                            │
-- │  Source: samples.tpch.lineitem (30M rows)                               │
-- │  Joins to: supplier_dim_mv (the canonical dimension MV!)                │
-- └─────────────────────────────────────────────────────────────────────────┘

CREATE OR REPLACE VIEW home_dipankar_kushari.metric_view.lineitem_fact_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Fact: Order line items (30M rows). Measures quantity and revenue.
  Joins to supplier_dim_mv (canonical dimension MV) for supplier/nation/region.
  NO dimension metadata is redefined here — it comes from the MV join.
source: samples.tpch.lineitem
joins:
  - name: supplier
    source: home_dipankar_kushari.metric_view.supplier_dim_mv
    on: "supplier.supplier_key = source.l_suppkey"
dimensions:
  - name: ship_date
    expr: l_shipdate
    comment: "Line item ship date"
    display_name: "Ship Date"
  - name: return_flag
    expr: l_returnflag
    comment: "Return flag (R=returned, A=accepted, N=none)"
    display_name: "Return Flag"
  - name: supplier_name
    expr: supplier.supplier_name
    comment: "From canonical supplier_dim_mv"
  - name: supplier_nation
    expr: supplier.supplier_nation
    comment: "From supplier_dim_mv → nation_dim_mv"
  - name: supplier_region
    expr: supplier.supplier_region
    comment: "From supplier_dim_mv → nation_dim_mv → region_dim_mv"
measures:
  - name: total_quantity
    expr: SUM(l_quantity)
    comment: "Total units ordered"
    display_name: "Quantity Ordered"
  - name: total_revenue
    expr: SUM(l_extendedprice * (1 - l_discount))
    comment: "Net revenue after line discount"
    display_name: "Revenue"
  - name: avg_discount
    expr: AVG(l_discount)
    comment: "Average discount applied"
    display_name: "Avg Discount"
$$

result


In [0]:
%sql
-- ┌─────────────────────────────────────────────────────────────────────────┐
-- │  FACT MV 2: partsupp_fact_mv                                            │
-- │  Source: samples.tpch.partsupp (4M rows)                                │
-- │  Joins to: supplier_dim_mv (SAME canonical dimension MV!)               │
-- └─────────────────────────────────────────────────────────────────────────┘

CREATE OR REPLACE VIEW home_dipankar_kushari.metric_view.partsupp_fact_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Fact: Part-supplier inventory (4M rows). Measures available qty and cost.
  Joins to THE SAME supplier_dim_mv as lineitem_fact_mv.
  NO dimension metadata is redefined — supplier/nation/region come from the MV.
source: samples.tpch.partsupp
joins:
  - name: supplier
    source: home_dipankar_kushari.metric_view.supplier_dim_mv
    on: "supplier.supplier_key = source.ps_suppkey"
dimensions:
  - name: supplier_name
    expr: supplier.supplier_name
    comment: "From canonical supplier_dim_mv"
  - name: supplier_nation
    expr: supplier.supplier_nation
    comment: "From supplier_dim_mv → nation_dim_mv"
  - name: supplier_region
    expr: supplier.supplier_region
    comment: "From supplier_dim_mv → nation_dim_mv → region_dim_mv"
measures:
  - name: total_availqty
    expr: SUM(ps_availqty)
    comment: "Total parts available in stock"
    display_name: "Available Qty"
  - name: total_supplycost
    expr: SUM(ps_supplycost)
    comment: "Total supply cost"
    display_name: "Supply Cost"
  - name: avg_supplycost
    expr: AVG(ps_supplycost)
    comment: "Average cost per part-supplier pair"
    display_name: "Avg Supply Cost"
$$

result


In [0]:
%sql
-- ============================================================================
-- Verify Layer 2: Query the fact MVs — dimension fields resolve from MV chain
-- ============================================================================

-- Query lineitem_fact_mv: revenue by supplier region
-- The engine resolves: lineitem → supplier_dim_mv → nation_dim_mv → region_dim_mv
SELECT supplier_region,
       MEASURE(total_quantity) AS qty_ordered,
       MEASURE(total_revenue) AS revenue
FROM home_dipankar_kushari.metric_view.lineitem_fact_mv
GROUP BY 1
ORDER BY revenue DESC

supplier_region,qty_ordered,revenue
ASIA,153959372.00,219077128551.1493
AMERICA,153694799.00,218866376888.6913
EUROPE,152951637.00,217974976043.3280
MIDDLE EAST,152861379.00,217806365817.0509
AFRICA,151625171.00,216110331946.9960


In [0]:
%sql
-- Query partsupp_fact_mv: supply cost by supplier region
-- Uses the EXACT same canonical dimension chain — zero metadata duplication!
SELECT supplier_region,
       MEASURE(total_availqty) AS qty_in_stock,
       MEASURE(total_supplycost) AS supply_cost
FROM home_dipankar_kushari.metric_view.partsupp_fact_mv
GROUP BY 1
ORDER BY supply_cost DESC

supplier_region,qty_in_stock,supply_cost
ASIA,4023280970,402624807.55
AMERICA,4018484428,402003912.12
EUROPE,4004157411,400561946.15
MIDDLE EAST,3998359672,400220834.61
AFRICA,3964366364,396810743.69


In [0]:
%sql
-- ============================================================================
-- LAYER 3: COMBINED MULTI-FACT MV (Canonical MV as SOURCE + ONE_TO_MANY)
-- ============================================================================
-- Best Practice #8: "Use the shared dimension as source with one_to_many joins"
-- 
-- The SOURCE is the canonical supplier_dim_mv itself (an MV, not a table!).
-- Both fact tables are joined as cardinality: one_to_many.
-- This gives a SINGLE metric view combining measures from both facts.
--
-- NOTE: cardinality: one_to_many requires DBR 18.1+.
-- ============================================================================

CREATE OR REPLACE VIEW home_dipankar_kushari.metric_view.supplier_orders_vs_supply_mv
WITH METRICS LANGUAGE YAML AS $$
version: "1.1"
comment: >
  Multi-fact metric view: Orders vs. Supply by supplier.
  SOURCE is the canonical supplier_dim_mv (MV, not the raw table!).
  Both fact tables join as one_to_many.
  All dimension metadata inherited from supplier_dim_mv chain.
  Demonstrates Best Practices #4 + #8 combined.
source: home_dipankar_kushari.metric_view.supplier_dim_mv
joins:
  - name: lineitem
    source: samples.tpch.lineitem
    on: "lineitem.l_suppkey = source.supplier_key"
    cardinality: one_to_many
  - name: partsupp
    source: samples.tpch.partsupp
    on: "partsupp.ps_suppkey = source.supplier_key"
    cardinality: one_to_many
dimensions:
  - name: supplier_name
    expr: supplier_name
    comment: "Inherited from supplier_dim_mv source"
  - name: supplier_nation
    expr: supplier_nation
    comment: "Inherited from supplier_dim_mv → nation_dim_mv"
  - name: supplier_region
    expr: supplier_region
    comment: "Inherited from supplier_dim_mv → nation_dim_mv → region_dim_mv"
measures:
  - name: qty_ordered
    expr: SUM(lineitem.l_quantity)
    comment: "Total units ordered (from lineitem fact)"
    display_name: "Qty Ordered"
  - name: order_revenue
    expr: SUM(lineitem.l_extendedprice * (1 - lineitem.l_discount))
    comment: "Net revenue from orders (from lineitem fact)"
    display_name: "Order Revenue"
  - name: qty_in_stock
    expr: SUM(partsupp.ps_availqty)
    comment: "Total available quantity (from partsupp fact)"
    display_name: "Qty In Stock"
  - name: supply_cost
    expr: SUM(partsupp.ps_supplycost)
    comment: "Total supply cost (from partsupp fact)"
    display_name: "Supply Cost"
$$

In [0]:
%sql
-- ============================================================================
-- PROOF: The centralization works — update the canonical MV, all facts benefit
-- ============================================================================
-- 
-- Let's verify the object dependency chain by describing the metric views:

DESCRIBE TABLE EXTENDED home_dipankar_kushari.metric_view.lineitem_fact_mv

col_name,data_type,comment,metadata
ship_date,date,Line item ship date,"{""display_name"":""Ship Date""}"
return_flag,string,"Return flag (R=returned, A=accepted, N=none)","{""display_name"":""Return Flag""}"
supplier_name,string,From canonical supplier_dim_mv,null
supplier_nation,string,From supplier_dim_mv → nation_dim_mv,null
supplier_region,string,From supplier_dim_mv → nation_dim_mv → region_dim_mv,null
total_quantity,"decimal(28,2) measure",Total units ordered,"{""display_name"":""Quantity Ordered""}"
total_revenue,"decimal(38,4) measure",Net revenue after line discount,"{""display_name"":""Revenue""}"
avg_discount,"decimal(22,6) measure",Average discount applied,"{""display_name"":""Avg Discount""}"
,,,
# Detailed Table Information,,,


In [0]:
%sql
-- ============================================================================
-- BONUS: Describe the supplier_dim_mv to see its resolved schema
-- ============================================================================

DESCRIBE TABLE EXTENDED home_dipankar_kushari.metric_view.supplier_dim_mv

col_name,data_type,comment,metadata
supplier_key,bigint,Primary key for supplier,null
supplier_name,string,Supplier name,"{""display_name"":""Supplier"",""synonyms"":[""vendor"",""supplier name""]}"
supplier_nation,string,Supplier's country — from nation_dim_mv,"{""display_name"":""Supplier Nation""}"
supplier_region,string,Supplier's region — from nation_dim_mv → region_dim_mv snowflake,"{""display_name"":""Supplier Region""}"
account_balance,"decimal(18,2)",Supplier account balance,"{""display_name"":""Account Balance""}"
,,,
# Detailed Table Information,,,
Catalog,home_dipankar_kushari,,
Database,metric_view,,
Table,supplier_dim_mv,,


In [0]:
%sql
-- ============================================================================
-- VERIFY: List all metric views we created
-- ============================================================================

USE CATALOG home_dipankar_kushari;
SHOW VIEWS IN metric_view

namespace,viewName,isTemporary,isMaterialized,isMetric
metric_view,lineitem_fact_mv,false,false,true
metric_view,nation_dim_mv,false,false,true
metric_view,partsupp_fact_mv,false,false,true
metric_view,region_dim_mv,false,false,true
metric_view,supplier_dim_mv,false,false,true
,_sqldf,true,false,false


In [0]:
%sql
-- ============================================================================
-- BONUS: Query both facts through their shared dimension to compare
-- ============================================================================
-- This simulates what the multi-fact MV (Layer 3) would do on DBR 18.1+.
-- Here we do it manually with CTEs to prove the numbers are correct.

WITH orders_by_nation AS (
    SELECT supplier_nation,
           MEASURE(total_quantity) AS qty_ordered,
           MEASURE(total_revenue) AS revenue
    FROM home_dipankar_kushari.metric_view.lineitem_fact_mv
    GROUP BY 1
),
supply_by_nation AS (
    SELECT supplier_nation,
           MEASURE(total_availqty) AS qty_in_stock,
           MEASURE(total_supplycost) AS supply_cost
    FROM home_dipankar_kushari.metric_view.partsupp_fact_mv
    GROUP BY 1
)
SELECT o.supplier_nation,
       o.qty_ordered,
       s.qty_in_stock,
       o.revenue,
       s.supply_cost
FROM orders_by_nation o
JOIN supply_by_nation s ON o.supplier_nation = s.supplier_nation
ORDER BY o.supplier_nation

supplier_nation,qty_ordered,qty_in_stock,revenue,supply_cost
ALGERIA,30118161.00,787747970,42860533624.3939,78444048.23
ARGENTINA,31379451.00,819235537,44620544239.5775,81990077.94
BRAZIL,30251738.00,792266106,43182069714.8985,79127627.26
CANADA,31015711.00,813361927,44183766734.8636,81278659.31
CHINA,30365550.00,794056484,43349774161.8894,79661603.82
EGYPT,30650041.00,801099509,43687673478.2822,80148806.02
ETHIOPIA,30051814.00,785167373,42768600735.4326,78705152.32
FRANCE,30394033.00,797342745,43225230224.3119,79735698.71
GERMANY,30671090.00,802898871,43656405351.1873,80202588.73
INDIA,31629388.00,825740578,44978532738.6463,82655327.38


In [0]:
%sql
-- ============================================================================
-- BONUS: Query both facts through their shared dimension to compare
-- ============================================================================
-- This simulates what the multi-fact MV (Layer 3) would do on DBR 18.1+.
-- Here we do it manually with CTEs to prove the numbers are correct.

WITH orders_by_nation AS (
    SELECT supplier_nation,
           MEASURE(total_quantity) AS qty_ordered,
           MEASURE(total_revenue) AS revenue
    FROM home_dipankar_kushari.metric_view.lineitem_fact_mv
    GROUP BY 1
),
supply_by_nation AS (
    SELECT supplier_nation,
           MEASURE(total_availqty) AS qty_in_stock,
           MEASURE(total_supplycost) AS supply_cost
    FROM home_dipankar_kushari.metric_view.partsupp_fact_mv
    GROUP BY 1
)
SELECT o.supplier_nation,
       o.qty_ordered,
       s.qty_in_stock,
       o.revenue,
       s.supply_cost
FROM orders_by_nation o
JOIN supply_by_nation s ON o.supplier_nation = s.supplier_nation
ORDER BY o.supplier_nation

supplier_nation,qty_ordered,qty_in_stock,revenue,supply_cost
ALGERIA,30118161.00,787747970,42860533624.3939,78444048.23
ARGENTINA,31379451.00,819235537,44620544239.5775,81990077.94
BRAZIL,30251738.00,792266106,43182069714.8985,79127627.26
CANADA,31015711.00,813361927,44183766734.8636,81278659.31
CHINA,30365550.00,794056484,43349774161.8894,79661603.82
EGYPT,30650041.00,801099509,43687673478.2822,80148806.02
ETHIOPIA,30051814.00,785167373,42768600735.4326,78705152.32
FRANCE,30394033.00,797342745,43225230224.3119,79735698.71
GERMANY,30671090.00,802898871,43656405351.1873,80202588.73
INDIA,31629388.00,825740578,44978532738.6463,82655327.38


In [0]:
%sql
-- Verify Layer 3: Query the combined multi-fact MV
SELECT supplier_nation,
       MEASURE(qty_ordered)   AS qty_ordered,
       MEASURE(qty_in_stock)  AS qty_in_stock,
       MEASURE(order_revenue) AS revenue,
       MEASURE(supply_cost)   AS cost
FROM home_dipankar_kushari.metric_view.supplier_orders_vs_supply_mv
GROUP BY 1
ORDER BY 1;

supplier_nation,qty_ordered,qty_in_stock,revenue,cost
ALGERIA,30118161.00,787747970,42860533624.3939,78444048.23
ARGENTINA,31379451.00,819235537,44620544239.5775,81990077.94
BRAZIL,30251738.00,792266106,43182069714.8985,79127627.26
CANADA,31015711.00,813361927,44183766734.8636,81278659.31
CHINA,30365550.00,794056484,43349774161.8894,79661603.82
EGYPT,30650041.00,801099509,43687673478.2822,80148806.02
ETHIOPIA,30051814.00,785167373,42768600735.4326,78705152.32
FRANCE,30394033.00,797342745,43225230224.3119,79735698.71
GERMANY,30671090.00,802898871,43656405351.1873,80202588.73
INDIA,31629388.00,825740578,44978532738.6463,82655327.38
